# Fabric visibility from spectra and photos

1. Open this notebook in Colab and run the Drive cell.
2. Edit **Settings**, then run all cells below it. Start with **Fit** on the original dataset.
3. For a new dataset, select **Apply**, change its input paths and output name, and keep the saved model path. Supply photos to measure prediction error, or set both photo settings to `None` to generate predictions from spectra alone.

The four spectral workbooks use the first column for sample names and the remaining columns for wavelengths. Files may use the original `REFLECT05` to `REFLECT35` headers, or explicit numeric wavelengths in nm. Excel reports include the image actually matched to each sample.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 1. Settings

**Legacy weighting** reproduces the original 55-sample report. It pairs the first 31 values after the header in the photopic workbook with the D65 values by row position. **Wavelength weighting** matches D65 and photopic sensitivity by wavelength and requires its own fit. Keep the weighting setting and spectral measurement protocol consistent between Fit and Apply.

The source spectra in the original workbooks use percentages. For sources stored as fractions from 0 to 1, set the corresponding unit to `fraction`. Direct transmittance is always passed to the model in percentage points. Diffuseness and opacity are ratios.

In [ ]:
from pathlib import Path
from IPython.display import display

ROOT = Path("/content/drive/MyDrive/Master")
SPECTRA = ROOT / "Data 9th September"  # Point this at the folder with the four All workbooks.

MODE = "Fit"  # "Fit" creates the model. "Apply" loads the saved model.
FILES = {
    "total": SPECTRA / "All Transmittance on white background.xlsx",
    "diffuse": SPECTRA / "All Transmittance on black background.xlsx",
    "white": SPECTRA / "All Reflection on white background.xlsx",
    "black": SPECTRA / "All reflection on black background.xlsx",
}
PHOTOPIC_FILE = ROOT / "CIE_sle_photopic (1).xlsx"
D65_FILE = ROOT / "D65.xlsx"
WEIGHTING = "legacy"  # "legacy" or "wavelength"
TRANSMISSION_UNIT = "percent"
REFLECTION_UNIT = "percent"

IMAGE_FOLDER = ROOT / "All images"
REFERENCE_IMAGE = IMAGE_FOLDER / "No fabric cabinet.JPG"
# For a filename that differs from its sample name, enter the exact JPEG filename.
PHOTO_NAMES = {}  # Example: {"Sample 01": "Photo_001.JPG"}

# Fit on the original 55 samples with the same groups as the previous model.
# For a different training dataset, choose "None" or supply a family for every sample.
FAMILY_GROUPS = "original"  # "original", None, or a dict of {sample name: family}

OUTPUT_FOLDER = ROOT / "Visibility workflow"
REPORT_NAME = "original_visibility.xlsx"  # Choose a different name when applying the model.
MODEL_FILE = OUTPUT_FOLDER / "original_model.json"

## 2. Calculate optical and image metrics

Rows are matched by sample name. Photos are matched by exact sample name after accounting for spacing and cabinet/backing suffixes. The measured image metrics use the same crop, grayscale conversion, normalization, and equations as the original combined report.

In [ ]:
import json
import re
import tempfile

import cv2
import numpy as np
import pandas as pd
from scipy.optimize import least_squares
from skimage.metrics import structural_similarity


PREDICTORS = ["weighted_T_dir", "diffuseness", "opacity"]
TARGETS = ["SSIM", "Acutance", "Michelson_Contrast"]
ORIGINAL_WAVELENGTHS = np.arange(400, 701, 10)
EXPONENT_BOUNDS = (-5.0, 5.0)
CROP = (0.25, 0.52, 0.50, 0.40)
MODEL_VERSION = 1


def sample_key(label):
    name = str(label).strip().casefold()
    name = re.sub(r"\s*cabinet$", "", name).strip()
    name = re.sub(r"(?:\s+|(?<=\d))on\s+(white|black)$", "", name).strip()
    name = re.sub(r"\s+nonwoven$", "", name)
    name = re.sub(r"\+\s*1\s*(?=[a-z])", "+", name)
    return re.sub(r"\s+", "", name)


def wavelength_columns(columns, path):
    names = [str(value).strip().upper() for value in columns]
    if set(names) == {f"REFLECT{i:02d}" for i in range(5, 36)}:
        return np.array([400 + 10 * (int(name[7:]) - 5) for name in names])
    try:
        values = np.array([float(re.sub(r"\s*NM$", "", name)) for name in names])
    except ValueError as error:
        raise ValueError(f"{path.name} needs REFLECT05–35 or numeric wavelength headers.") from error
    if not np.isfinite(values).all() or len(set(values)) != len(values):
        raise ValueError(f"Invalid or repeated wavelength headers in {path.name}.")
    return values


def read_spectra(path, unit):
    path = Path(path)
    source = pd.read_excel(path, index_col=0).dropna(how="all")
    labels = source.index.to_series()
    keys = labels.map(sample_key)
    if source.empty or labels.isna().any() or keys.eq("").any() or keys.duplicated().any():
        raise ValueError(f"Missing or duplicate sample names in {path.name}.")
    wavelengths = wavelength_columns(source.columns, path)
    values = source.apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
    if not np.isfinite(values).all():
        row, col = np.argwhere(~np.isfinite(values))[0]
        raise ValueError(f"{path.name}: missing or invalid value for {labels.iloc[row]} at {wavelengths[col]:g} nm.")
    if unit not in {"percent", "fraction"}:
        raise ValueError("Use percent or fraction for each spectral unit setting.")
    if unit == "fraction" and np.abs(values).max() > 1:
        raise ValueError(f"{path.name}: values exceed the declared fraction scale.")
    values = values * 100 if unit == "fraction" else values
    return pd.Series(labels.to_numpy(), index=keys), pd.DataFrame(values, index=keys, columns=wavelengths)


def at_wavelengths(path, wavelengths):
    rows = pd.read_excel(path, header=None, usecols=[0, 1]).apply(pd.to_numeric, errors="coerce")
    data = rows.dropna(how="all").dropna(subset=[0])
    if data[0].duplicated().any() or not np.isfinite(data.to_numpy()).all():
        raise ValueError(f"Invalid or duplicate weighting entries in {Path(path).name}.")
    selected = data.set_index(0)[1].reindex(wavelengths)
    if selected.isna().any():
        raise ValueError(f"{Path(path).name} is missing wavelengths {selected.index[selected.isna()].tolist()}.")
    return selected.to_numpy()


def spectral_weights(wavelengths):
    if WEIGHTING == "legacy":
        if not np.array_equal(wavelengths, ORIGINAL_WAVELENGTHS):
            raise ValueError("Legacy weighting requires the original 400–700 nm grid at 10 nm steps.")
        d65 = pd.read_excel(D65_FILE).iloc[:31, 1].to_numpy(dtype=float)
        photopic = pd.read_excel(PHOTOPIC_FILE).iloc[:31, 1].to_numpy(dtype=float)
        if d65.size != 31 or photopic.size != 31:
            raise ValueError("Legacy weighting needs 31 D65 and photopic values after the header.")
    elif WEIGHTING == "wavelength":
        d65 = at_wavelengths(D65_FILE, wavelengths)
        photopic = at_wavelengths(PHOTOPIC_FILE, wavelengths)
    else:
        raise ValueError("Set WEIGHTING to legacy or wavelength.")
    weights = d65 * photopic
    if not np.isfinite(weights).all() or (weights < 0).any() or weights.sum() <= 0:
        raise ValueError("Invalid combined D65 and photopic weights.")
    return weights


def compute_optics():
    units = {"total": TRANSMISSION_UNIT, "diffuse": TRANSMISSION_UNIT,
             "white": REFLECTION_UNIT, "black": REFLECTION_UNIT}
    sources = {name: read_spectra(FILES[name], unit) for name, unit in units.items()}
    labels, total = sources["total"]
    wavelengths = np.sort(total.columns.to_numpy())
    if wavelengths.size < 2 or not np.allclose(np.diff(wavelengths), np.diff(wavelengths)[0]):
        raise ValueError("Use a uniformly spaced wavelength grid for the original weighted-average formula.")
    weights = spectral_weights(wavelengths)
    weighted = {}
    for name, (_, data) in sources.items():
        missing = total.index.difference(data.index)
        extra = data.index.difference(total.index)
        if len(missing) or len(extra):
            raise ValueError(f"{name} spectrum sample mismatch. Missing {list(missing)}. Extra {list(extra)}.")
        if set(data.columns) != set(wavelengths):
            raise ValueError(f"The {name} spectrum wavelength grid differs from total transmission.")
        weighted[name] = (data.loc[total.index, wavelengths].to_numpy() @ weights) / weights.sum()

    report = pd.DataFrame({
        "Sample": labels.astype(str).str.strip(),
        "weighted_T_tot": weighted["total"],
        "weighted_T_diff": weighted["diffuse"],
        "weighted_T_dir": weighted["total"] - weighted["diffuse"],
        "weighted_R_w": weighted["white"],
        "weighted_R_b": weighted["black"],
    })
    report["diffuseness"] = report.weighted_T_diff / report.weighted_T_tot.where(report.weighted_T_tot > 0)
    report["opacity"] = report.weighted_R_b / report.weighted_R_w.where(report.weighted_R_w > 0)
    issues = []
    for row in report.itertuples():
        flags = []
        if row.weighted_T_tot <= 0 or row.weighted_R_w <= 0:
            flags.append("Nonpositive ratio denominator")
        if row.weighted_T_dir < 0:
            flags.append("Diffuse transmission exceeds total")
        if any(v < 0 or v > 100 for v in (row.weighted_T_tot, row.weighted_T_diff,
                                          row.weighted_R_w, row.weighted_R_b)):
            flags.append("Weighted spectrum outside 0–100%")
        if np.isfinite(row.opacity) and not 0 <= row.opacity <= 1:
            flags.append("Opacity outside 0–1")
        issues.append("; ".join(flags))
    report["Optical check"] = issues
    settings = {"weighting": WEIGHTING, "wavelengths": wavelengths.tolist(), "weights": (weights / weights.sum()).tolist(),
                "crop": list(CROP)}
    return report, settings


def read_gray(path):
    image = cv2.imread(str(path))
    if image is None:
        raise ValueError(f"Cannot read image {path}.")
    return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY).astype(np.float64)


def compute_images(report):
    report = report.copy()
    report["Image"] = ""
    report[TARGETS] = np.nan
    if IMAGE_FOLDER is None and REFERENCE_IMAGE is None:
        if MODE == "Fit":
            raise ValueError("Fit needs a reference and sample photos to calculate the measured targets.")
        return report
    if IMAGE_FOLDER is None or REFERENCE_IMAGE is None:
        raise ValueError("Set both photo settings, or set both to None.")
    folder, reference = Path(IMAGE_FOLDER), Path(REFERENCE_IMAGE)
    ref = read_gray(reference)
    height, width = ref.shape
    x, y, w, h = [round(size * proportion) for size, proportion in zip((width, height, width, height), CROP)]
    roi = np.s_[y:y+h, x:x+w]
    ref = ref[roi] / 255.0
    if min(ref.shape) < 11 or np.ptp(ref) <= 0:
        raise ValueError("The reference crop is too small or has no contrast.")
    ref /= ref.mean()

    candidates = {}
    for path in sorted(folder.iterdir()):
        if path.is_file() and path.suffix.casefold() in {".jpg", ".jpeg", ".png"} and path.resolve() != reference.resolve():
            candidates.setdefault(sample_key(path.stem), []).append(path)
    overrides = {sample_key(name): folder / filename for name, filename in PHOTO_NAMES.items()}
    if len(overrides) != len(PHOTO_NAMES) or not set(overrides) <= set(report.index):
        raise ValueError("Check PHOTO_NAMES for duplicate or unknown sample names.")
    matched = {}
    problems = []
    for key, label in report.Sample.items():
        paths = [overrides[key]] if key in overrides else candidates.get(key, [])
        if len(paths) != 1 or not paths[0].is_file() or paths[0].resolve() == reference.resolve():
            problems.append(f"{label}: " + (", ".join(p.name for p in paths) or "no matching image"))
        else:
            matched[key] = paths[0]
    if problems:
        raise ValueError("Set the exact filenames in PHOTO_NAMES:\n" + "\n".join(problems))
    if len({path.resolve() for path in matched.values()}) != len(matched):
        raise ValueError("More than one sample maps to the same image. Check PHOTO_NAMES.")

    for key, path in matched.items():
        crop = read_gray(path)[roi] / 255.0
        if crop.shape != ref.shape or crop.mean() <= 0:
            raise ValueError(f"{path.name}: the reference crop is incomplete or entirely black.")
        normalized = crop / crop.mean()
        ssim = structural_similarity(normalized, ref, data_range=np.ptp(ref),
                                     gaussian_weights=True, sigma=1.5, use_sample_covariance=False)
        gx = cv2.Sobel(crop, cv2.CV_64F, 1, 0, ksize=3)
        gy = cv2.Sobel(crop, cv2.CV_64F, 0, 1, ksize=3)
        acutance = np.sqrt(gx**2 + gy**2).mean()
        contrast = np.ptp(crop) / (crop.max() + crop.min())
        report.loc[key, "Image"] = path.name
        report.loc[key, TARGETS] = [ssim, acutance, contrast]
    return report

## 3. Fit or apply the existing models

Fit uses linear regression with an intercept and the original three-term power-sum with 20 seeded random starts plus the all-ones start and bounds of −5 to +5. It saves full-precision parameters. When family groups are available, the report includes predictions from fits that held out each family in turn. Apply keeps the fitted parameters fixed and flags optical inputs outside the training range.

In [ ]:
def fit_models(x, y):
    linear = np.linalg.lstsq(np.column_stack([np.ones(len(x)), x]), y, rcond=None)[0]
    scale = x.mean(axis=0)
    normalized = x / scale

    def design(values, exponents):
        return np.column_stack([values[:, i] ** exponents[i] for i in range(values.shape[1])])

    def solve(exponents):
        design_matrix = design(normalized, exponents)
        coefficients = np.linalg.lstsq(design_matrix, y, rcond=None)[0]
        return design_matrix, coefficients

    def residual(exponents):
        design, coefficients = solve(exponents)
        return y - design @ coefficients

    rng = np.random.default_rng(0)
    starts = [np.ones(x.shape[1]), *rng.uniform(-3, 3, size=(20, x.shape[1]))]
    best = None
    for start in starts:
        try:
            fit = least_squares(residual, start, bounds=EXPONENT_BOUNDS, method="trf")
        except (ValueError, FloatingPointError, np.linalg.LinAlgError):
            continue
        if np.isfinite(fit.cost) and (best is None or fit.cost < best.cost):
            best = fit
    if best is None:
        raise RuntimeError("All starts failed for the power-sum fit.")
    _, coefficients = solve(best.x)
    if not np.isfinite(coefficients).all():
        raise RuntimeError("The power-sum fit produced non-finite coefficients.")
    return {"Linear": {"coefficients": linear.tolist()},
            "Powersum": {"coefficients": coefficients.tolist(), "exponents": best.x.tolist(),
                         "scale": scale.tolist()}}


def predict_models(models, x):
    linear = np.full(len(x), np.nan)
    power = np.full(len(x), np.nan)
    finite = np.isfinite(x).all(axis=1)
    positive = finite & (x > 0).all(axis=1)
    linear[finite] = np.column_stack([np.ones(finite.sum()), x[finite]]) @ models["Linear"]["coefficients"]
    fit = models["Powersum"]
    with np.errstate(over="ignore", invalid="ignore"):
        scaled = x[positive] / fit["scale"]
        design = np.column_stack([scaled[:, i] ** e for i, e in enumerate(fit["exponents"])])
        power[positive] = design @ fit["coefficients"]
    return {"Linear": np.where(np.isfinite(linear), linear, np.nan),
            "Powersum": np.where(np.isfinite(power), power, np.nan)}


def training_families(report):
    if FAMILY_GROUPS is None:
        return None
    if isinstance(FAMILY_GROUPS, str) and FAMILY_GROUPS == "original":
        labels = report.Sample.str.lower()
        groups = labels.str.split().str[0].mask(labels.str.match(r"^\d"), "mixed")
        counts = groups.value_counts().to_dict()
        expected = {"yellow": 11, "blue": 11, "white": 9, "red": 9, "brown": 9, "mixed": 6}
        if counts != expected:
            raise ValueError("Original family grouping expects the original 55 samples. Supply FAMILY_GROUPS or use None.")
        return groups
    if not isinstance(FAMILY_GROUPS, dict):
        raise ValueError("Use original, None, or a sample-to-family dictionary for FAMILY_GROUPS.")
    mapping = {sample_key(name): str(family).strip() for name, family in FAMILY_GROUPS.items()}
    if len(mapping) != len(FAMILY_GROUPS) or set(mapping) != set(report.index) or not all(mapping.values()):
        raise ValueError("FAMILY_GROUPS must give exactly one family for each training sample.")
    return pd.Series(mapping).reindex(report.index)


def rmse_rows(target, label, observed, predictions):
    actual = np.asarray(observed)
    shared = np.isfinite(actual) & np.logical_and.reduce([np.isfinite(p) for p in predictions.values()])
    rows = []
    for name, prediction in predictions.items():
        valid = np.isfinite(actual) & np.isfinite(prediction)
        errors = actual[shared] - prediction[shared]
        rows.append([target, "Power-sum" if name == "Powersum" else name, label, int(valid.sum()), int(shared.sum()),
                     np.sqrt(np.mean(errors**2)) if len(errors) else None,
                     np.mean(errors) if len(errors) else None])
    return rows


def validate_model(bundle, settings):
    if bundle.get("version") != MODEL_VERSION or bundle.get("predictors") != PREDICTORS:
        raise ValueError("This saved model uses an incompatible format.")
    trained = bundle["preprocessing"]
    for key in ("weighting", "wavelengths", "crop"):
        if trained[key] != settings[key]:
            raise ValueError(f"Saved model and current preprocessing differ in {key}. Use the original settings.")
    if not np.allclose(trained["weights"], settings["weights"], rtol=1e-10, atol=1e-12):
        raise ValueError("The effective D65 and photopic weights differ from those used to fit this model.")


def run_models(report, settings):
    x = report[PREDICTORS].to_numpy(dtype=float)
    families = training_families(report) if MODE == "Fit" else None
    if MODE == "Fit":
        if len(x) <= len(PREDICTORS) or not np.isfinite(x).all() or (x <= 0).any():
            invalid = report.loc[~(np.isfinite(x) & (x > 0)).all(axis=1), "Sample"].tolist()
            raise ValueError(f"Fit needs at least four complete samples with positive predictors. Check {invalid}.")
        bundle = {"version": MODEL_VERSION, "predictors": PREDICTORS, "preprocessing": settings,
                  "training_samples": report.Sample.tolist(),
                  "training_min": x.min(axis=0).tolist(), "training_max": x.max(axis=0).tolist(), "fits": {}}
    else:
        bundle = json.loads(Path(MODEL_FILE).read_text(encoding="utf-8"))
        validate_model(bundle, settings)
        outside = (x < bundle["training_min"]) | (x > bundle["training_max"])
        report["Outside training range"] = [", ".join(np.array(PREDICTORS)[flags]) for flags in outside]
        report["Prediction check"] = [
            "Invalid predictor" if not np.isfinite(row).all() else
            "Power-sum requires positive predictors" if (row <= 0).any() else "" for row in x
        ]

    predictions, scores = {}, []
    for target in TARGETS:
        y = report[target].to_numpy(dtype=float)
        if MODE == "Fit":
            if not np.isfinite(y).all():
                raise ValueError(f"Missing or invalid measured values for {target}.")
            bundle["fits"][target] = fit_models(x, y)
        values = predict_models(bundle["fits"][target], x)
        label = "Training fit" if MODE == "Fit" else "Apply"
        scores.extend(rmse_rows(target, label, y, values))
        pred = report[["Sample"]].copy()
        if families is not None:
            pred["Family"] = families
        pred["Actual"] = y
        suffix = "all_data" if MODE == "Fit" else "saved_model"
        for name, value in values.items():
            pred[f"{name}_{suffix}"] = value
            if MODE == "Apply":
                pred[f"{name}_residual"] = y - value
        if families is not None:
            grouped = {name: np.full(len(y), np.nan) for name in values}
            for family in families.unique():
                held_out = families.eq(family).to_numpy()
                if (~held_out).sum() <= len(PREDICTORS):
                    raise ValueError(f"Too few training samples when family {family} is held out.")
                fold_models = fit_models(x[~held_out], y[~held_out])
                for name, value in predict_models(fold_models, x[held_out]).items():
                    grouped[name][held_out] = value
            scores.extend(rmse_rows(target, "Family held out", y, grouped))
            for name, value in grouped.items():
                pred[f"{name}_grouped_cv"] = value
        predictions[target] = pred
    columns = ["Image metric", "Model", "Evaluation", "Scored samples", "Shared samples", "Shared RMSE", "Shared mean error"]
    return bundle, pd.DataFrame(scores, columns=columns), predictions

## 4. Run and save

Running Fit replaces the model at MODEL_FILE. Use a new model filename for a different fit.

Open **Summary** first. Its RMSE and mean error compare the models on the same measured samples. **Per-Sample Values** contains the measurements and input checks. The **Pred** sheets contain predictions for each image metric. The JSON file holds the fitted model for later use in Apply mode.

When Apply uses spectra alone, the measured values and errors are blank. A new dataset can use the same workflow with different file paths. The three prediction sheets still contain estimates where the optical inputs permit them.

In [ ]:
def save_report(report, summary, predictions, settings):
    destination = Path(OUTPUT_FOLDER) / REPORT_NAME
    if destination.suffix.lower() != ".xlsx":
        raise ValueError("REPORT_NAME must end in .xlsx.")
    sources = [*FILES.values(), D65_FILE, PHOTOPIC_FILE, MODEL_FILE]
    if destination.resolve() in {Path(path).resolve() for path in sources}:
        raise ValueError("Choose a report filename different from every input and the model.")
    destination.parent.mkdir(parents=True, exist_ok=True)
    notes = pd.DataFrame([
        ("Mode", MODE), ("Spectral weighting", settings["weighting"]),
        ("Model file", str(MODEL_FILE)), ("Reference", str(REFERENCE_IMAGE or "")),
        ("Photopic sensitivity", str(PHOTOPIC_FILE)), ("D65", str(D65_FILE)),
        ("Input transmission unit", TRANSMISSION_UNIT), ("Input reflection unit", REFLECTION_UNIT),
        *((f"{name} spectra", str(path)) for name, path in FILES.items()),
        ("Error sign", "Measured minus predicted"),
        ("Shared RMSE", "Calculated only on measured rows with predictions from both models."),
    ], columns=["Setting", "Value"])
    with tempfile.TemporaryDirectory(dir=destination.parent) as directory:
        temporary = Path(directory) / "report.xlsx"
        with pd.ExcelWriter(temporary, engine="openpyxl") as writer:
            sheets = {"Summary": summary, "Per-Sample Values": report,
                      **{f"Pred_{name}": data for name, data in predictions.items()}, "Inputs": notes}
            for name, data in sheets.items():
                data.to_excel(writer, sheet_name=name, index=False, freeze_panes=(1, 1))
                sheet = writer.sheets[name]
                sheet.auto_filter.ref = sheet.dimensions
                for cells in sheet.columns:
                    width = max(len(str(cell.value)) for cell in cells if cell.value is not None)
                    sheet.column_dimensions[cells[0].column_letter].width = min(width + 3, 52)
                    for cell in cells[1:]:
                        if isinstance(cell.value, float):
                            cell.number_format = "0.0000"
        temporary.replace(destination)
    return destination


if MODE not in {"Fit", "Apply"}:
    raise ValueError("Set MODE to Fit or Apply.")

if MODE == "Fit":
    model_path = Path(MODEL_FILE)
    original_paths = [*FILES.values(), D65_FILE, PHOTOPIC_FILE]
    if model_path.suffix.lower() != ".json" or model_path.resolve() in {Path(p).resolve() for p in original_paths}:
        raise ValueError("Choose a separate .json file for MODEL_FILE.")

measurements, preprocessing = compute_optics()
measurements = compute_images(measurements)
model, summary, predictions = run_models(measurements, preprocessing)
report_path = save_report(measurements, summary, predictions, preprocessing)
if MODE == "Fit":
    model_path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(dir=model_path.parent) as directory:
        temporary = Path(directory) / "model.json"
        temporary.write_text(json.dumps(model, indent=2, allow_nan=False), encoding="utf-8")
        temporary.replace(model_path)
    print(f"Saved model to {model_path}")
print(f"Saved report for {len(measurements)} samples to {report_path}")
display(summary.round(4))